# recs_023 -- Stage 3 Ablation B qualitative check (franchise/marketing over-indexing)

Manual read of `rag_chunk_v1_query_plus_desc` vs `rag_chunk_v1_raw_query` results, checking for
the franchise/marketing over-indexing failure mode flagged in `_scrap/rag_extension.md` before
trusting `query_plus_desc`'s win on `eval_retrieval_overall.csv`.

# Executive Summary

**Question:** Does appending the query game's IGDB description to the review text
(`rag_chunk_v1_query_plus_desc`) cause retrieval to over-index on franchise/studio/marketing
similarity instead of the thematic/sentiment similarity the reviewer described?

**Result:** The specific bias never appears -- across the 5 most-divergent examples (top-10
overlap = 0/10) with franchise- or company-tagged query games, no query game's own franchise or
company ID reappeared in either arm's top-5. But `query_plus_desc` is not a uniform genre-coherence
win either: 3/5 (Terraria, PUBG, Rust) show a clear improvement, 1/5 (NieR:Automata) is a clear
regression -- `raw_query` surfaces tighter action-RPG/hack-and-slash peers while `query_plus_desc`
drifts into strategy/tactical games -- and 1/5 (GTA V) is mixed.

**Recommendation / Decision:** Keep `query_plus_desc` as the Ablation B default -- the predicted
franchise/marketing bias did not materialize, and its net quantitative edge over `raw_query`
(`eval_retrieval_overall.csv`: Hit@K 0.475 vs 0.425) still holds. But treat "description helps"
as example-dependent, not universal -- the NieR:Automata regression is worth a larger sample
before leaning on this arm further.

# Business Context

Stage 3 wired 2 Ablation-B retrieval methods into the eval registry and ran them on the full val
cohort (`rag_v1` run, `docs/plans/rag_extension_plan.md`). `query_plus_desc` won on aggregate
Hit@K/Recall@K, but the origin plan (`_scrap/rag_extension.md`) flagged a specific risk: IGDB
descriptions are often genre-boilerplate/marketing copy, so appending them to the query could
pull retrieval toward "same franchise / same studio" matches rather than the thematic/sentiment
similarity a reviewer actually described -- and that pattern would look identical to a legitimate
hit in the aggregate metrics. Needs a manual read to tell the two apart before treating the
metric win as real.

# Research Question

**Research Question:** On examples where `rag_chunk_v1_raw_query` and
`rag_chunk_v1_query_plus_desc` retrieve the most different top-10 candidate sets, do
`query_plus_desc`'s results look like franchise/studio/marketing-copy matches, or like genuine
thematic/genre matches?

# Hypothesis

**Hypothesis:** `query_plus_desc`'s top results will show a higher rate of same-franchise or
same-studio matches to the query game than `raw_query`'s, consistent with the over-indexing
failure mode the origin plan called out.

**Success Criteria:** Franchise/company IDs on `query_plus_desc` results should not repeat the
query game's own franchise/company IDs more often than `raw_query`'s results do, on a sample of
the most-divergent examples.

**Result: not confirmed** -- no franchise/company ID repeats observed in either arm across the
sample. `query_plus_desc` shows tighter genre alignment in most (3/5) but not all examples --
NieR:Automata is a counter-case where `raw_query`'s genre match is actually tighter.

# Definitions

| Term | Definition | Notes |
|------|------------|-------|
| `overlap@10` | `\|top10(raw_query) ∩ top10(query_plus_desc)\|` for one example | 0 = fully divergent, 10 = identical |
| Ablation B | raw review text vs. review text + query game's own IGDB description, as the query text | `docs/plans/rag_extension_plan.md` decision #1 |
| franchise/marketing over-indexing | retrieval driven by shared IGDB `franchises`/`involved_companies` tags rather than thematic similarity | the failure mode this notebook checks for |

# Data Sources

## Data Source 1

**Source Name:** `eval_offline_examples.jsonl` (`rag_v1` run)

**Location:** `artifacts/recs/offline_eval/runs/rag_v1/eval_offline_examples.jsonl`

**Population Covered:** Full val cohort, 12,500 examples x 6 methods (this notebook uses only
the 2 `rag_chunk_v1_*` rows per example).

**Filters Applied:** None beyond what the `rag_v1` eval run already applied
(`configs/recs_job_eval_offline_rag_v1.json`).

**Known Limitations:** No query text stored (flagged in the plan doc as a Stage 4 gap) -- this
check reads retrieved-candidate genre/franchise/company metadata, not the literal review text.

---

## Data Source 2

**Source Name:** `igdb_games__enriched.parquet`

**Location:** `artifacts/igdb/igdb_games__enriched.parquet`

**Population Covered:** All catalog games with an IGDB join.

**Filters Applied:** None -- indexed by `app_id` for lookup only.

**Known Limitations:** `franchises`/`involved_companies` are IGDB numeric IDs, not names -- read
as presence/absence and cross-example ID repeats, not resolved to human-readable studio/franchise
names.

# Design / Process

**Methodology:** Manual qualitative read, not a new eval job -- reuses the already-run `rag_v1`
retrieval output.

**Analysis Approach**
1. Load `eval_offline_examples.jsonl`, keep the 2 `rag_chunk_v1_*` rows per `ex_idx`.
2. Compute top-10 overlap between the two arms for all 12,500 examples.
3. Sort by overlap ascending; keep only query games with `franchises` or `involved_companies` set
   in IGDB (franchise bias, if present, needs this metadata to be checkable).
4. Take the 5 most-divergent such examples; print each arm's top-5 with title/franchise/
   company/genre.
5. Read for a pattern: does `query_plus_desc` repeat the query game's own franchise/company, or
   show broader genre-level matches, vs. `raw_query`?

# Evaluation Outputs / Artifacts

| Artifact | Location | Description |
|---|---|---|
| `eval_retrieval_overall.csv` | `artifacts/recs/offline_eval/runs/rag_v1/` | Quantitative baseline this qualitative check sits on top of |
| `eval_retrieval_by_slice.csv` | `artifacts/recs/offline_eval/runs/rag_v1/` | Slice A/B breakdown referenced in Executive Summary |
| This notebook | `notebooks/retrieval/recs_023_stage3_qualitative_check.ipynb` | Qualitative read; no new artifact produced |

# Notebook Roadmap

1. Quantitative recap (`eval_retrieval_overall.csv`, by-slice)
2. Load Ablation-B examples, compute top-10 overlap per example
3. Select divergent examples with franchise/company-tagged query games
4. Side-by-side comparison, enriched with titles/franchise/company/genre
5. Key findings

# Analysis

## Setup

In [7]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


REPO_ROOT = _find_repo_root(Path.cwd())
RUN_DIR = REPO_ROOT / "artifacts" / "recs" / "offline_eval" / "runs" / "rag_v1"
RAW_QUERY = "rag_chunk_v1_raw_query"
QUERY_PLUS_DESC = "rag_chunk_v1_query_plus_desc"

pd.options.display.max_colwidth = 60

## Quantitative Recap

In [8]:
overall = pd.read_csv(RUN_DIR / "eval_retrieval_overall.csv")
by_slice = pd.read_csv(RUN_DIR / "eval_retrieval_by_slice.csv")

methods = [RAW_QUERY, QUERY_PLUS_DESC, "raw", "two_tower_v1"]
print("Overall:")
display(overall.set_index("method").loc[methods, ["Hit@K", "Precision@K", "Recall@K"]])

print("\nBy slice (primary metric: Recall@K for Slice A, Hit@K for Slice B):")
for slice_name in ["slice_a_multi_target", "slice_b_single_target"]:
    sub = by_slice[by_slice["slice_name"] == slice_name].set_index("method").loc[methods, ["Hit@K", "Recall@K"]]
    print(f"\n{slice_name}:")
    display(sub)

Overall:


,Hit@K,Precision@K,Recall@K
method,,,
rag_chunk_v1_raw_query,0.42480,0.004474,0.405454
rag_chunk_v1_query_plus_desc,0.47480,0.005036,0.456046
raw,0.45296,0.004801,0.434516
two_tower_v1,0.51224,0.005414,0.494053



By slice (primary metric: Recall@K for Slice A, Hit@K for Slice B):

slice_a_multi_target:


,Hit@K,Recall@K
method,,
rag_chunk_v1_raw_query,0.722759,0.389207
rag_chunk_v1_query_plus_desc,0.776552,0.453210
raw,0.747586,0.429578
two_tower_v1,0.773793,0.460228



slice_b_single_target:


,Hit@K,Recall@K
method,,
rag_chunk_v1_raw_query,0.406454,0.406454
rag_chunk_v1_query_plus_desc,0.456221,0.456221
raw,0.434820,0.434820
two_tower_v1,0.496136,0.496136


## Load Ablation-B Examples

In [9]:
by_ex: dict[int, dict[str, dict]] = {}
with open(RUN_DIR / "eval_offline_examples.jsonl") as f:
    for line in f:
        d = json.loads(line)
        if d["method"] in (RAW_QUERY, QUERY_PLUS_DESC):
            by_ex.setdefault(d["ex_idx"], {})[d["method"]] = d

n_both = sum(1 for m in by_ex.values() if RAW_QUERY in m and QUERY_PLUS_DESC in m)
print(f"examples with both methods: {n_both}")

examples with both methods: 12500


## Compute Top-10 Overlap

In [10]:
rows = []
for ex_idx, methods_d in by_ex.items():
    if RAW_QUERY not in methods_d or QUERY_PLUS_DESC not in methods_d:
        continue
    raw_top10 = json.loads(methods_d[RAW_QUERY]["retrieved_app_ids_json"])[:10]
    desc_top10 = json.loads(methods_d[QUERY_PLUS_DESC]["retrieved_app_ids_json"])[:10]
    overlap = len(set(raw_top10) & set(desc_top10))
    rows.append(
        dict(
            ex_idx=ex_idx,
            query_app_id=methods_d[RAW_QUERY]["query_app_id"],
            overlap=overlap,
            raw_top10=raw_top10,
            desc_top10=desc_top10,
        )
    )

overlap_df = pd.DataFrame(rows).sort_values("overlap")
print("overlap@10 distribution (0=fully divergent, 10=identical):")
print(overlap_df["overlap"].value_counts().sort_index())

overlap@10 distribution (0=fully divergent, 10=identical):
overlap
0     4801
1     1998
2     1305
3     1073
4      875
5      786
6      605
7      499
8      323
9      182
10      53
Name: count, dtype: int64


## Select Divergent, Franchise-Tagged Examples

In [11]:
igdb = pd.read_parquet(REPO_ROOT / "artifacts" / "igdb" / "igdb_games__enriched.parquet").set_index("app_id")


def field(app_id: int, col: str):
    if app_id not in igdb.index:
        return None
    v = igdb.loc[app_id][col]
    if isinstance(v, np.ndarray):
        return list(v) if v.size else None
    return None if pd.isna(v) else v


def info(app_id: int) -> dict:
    return dict(
        title=field(app_id, "app_name") or field(app_id, "igdb_name") or f"appid:{app_id}",
        franchises=field(app_id, "franchises"),
        companies=field(app_id, "involved_companies"),
        genres=field(app_id, "genres_names"),
    )


# Most-divergent examples where the query game itself carries franchise/company metadata --
# franchise bias, if present, needs this metadata to be checkable.
picked = []
for row in overlap_df.itertuples():
    qinfo = info(row.query_app_id)
    if qinfo["franchises"] or qinfo["companies"]:
        picked.append((row, qinfo))
    if len(picked) >= 5:
        break

print(f"selected {len(picked)} divergent examples with franchise/company-tagged query games")

selected 5 divergent examples with franchise/company-tagged query games


## Side-by-Side Comparison

In [12]:
for row, qinfo in picked:
    print(f"\n{'=' * 90}")
    print(f"ex_idx={row.ex_idx}  query_app_id={row.query_app_id}  overlap@10={row.overlap}")
    print(
        f"QUERY GAME: {qinfo['title']} | franchises={qinfo['franchises']} | "
        f"companies={qinfo['companies']} | genres={qinfo['genres']}"
    )
    print(f"-- {RAW_QUERY} top5 --")
    for a in row.raw_top10[:5]:
        i = info(a)
        print(f"   {a:>8}  {i['title']:<38} franchises={i['franchises']} companies={i['companies']} genres={i['genres']}")
    print(f"-- {QUERY_PLUS_DESC} top5 --")
    for a in row.desc_top10[:5]:
        i = info(a)
        print(f"   {a:>8}  {i['title']:<38} franchises={i['franchises']} companies={i['companies']} genres={i['genres']}")


ex_idx=12491  query_app_id=105600  overlap@10=0
QUERY GAME: Terraria | franchises=None | companies=[np.int64(16887), np.int64(189039), np.int64(188989), np.int64(189041), np.int64(289063), np.int64(289064), np.int64(289065)] | genres=['Platform', 'Role-playing (RPG)', 'Simulator', 'Strategy', 'Adventure', 'Indie']
-- rag_chunk_v1_raw_query top5 --
     823130  Totally Accurate Battlegrounds         franchises=None companies=[np.int64(65368)] genres=['Shooter', 'Indie']
    1118200  People Playground                      franchises=None companies=[np.int64(115528), np.int64(115529)] genres=['Simulator', 'Indie']
    1089980  The Henry Stickmin Collection          franchises=None companies=[np.int64(105468), np.int64(105469)] genres=['Point-and-click', 'Adventure', 'Indie']
     688130  Pogostuck: Rage With Your Friends      franchises=None companies=None genres=['Platform', 'Adventure', 'Indie']
     788260  Rules Of Survival                      franchises=None companies=[np.int64(293

# Key Findings

**Finding 1:** No franchise/marketing over-indexing observed. Across the 5 most-divergent
examples (overlap@10=0) with franchise/company-tagged query games (Terraria, PUBG, Rust,
NieR:Automata, GTA V), zero `query_plus_desc` (or `raw_query`) results repeated the query game's
own franchise or company ID. The specific failure mode the origin plan named -- retrieval
snapping to the query's own franchise/studio -- did not appear in either arm.

**Finding 2:** `query_plus_desc` is a genre-coherence win in most, not all, of the sample.
Terraria -> `raw_query` surfaces an unrelated indie mix (Totally Accurate Battlegrounds, Henry
Stickmin); `query_plus_desc` surfaces sandbox-survival genre-mates (Kenshi, Deep Rock Galactic,
RimWorld, Avorion). PUBG and Rust show the same pattern -- `query_plus_desc` lands on
survival/crafting peers (SCUM, Hunt: Showdown, Raft, Desolate) where `raw_query` lands on
unrelated genres.

**Finding 3 (counter-example):** NieR:Automata inverts the pattern. `raw_query` surfaces tight
action-RPG/hack-and-slash peers (Dragon Quest Heroes II, Devil May Cry HD, Toukiden 2, Chrono
Trigger); `query_plus_desc` drifts into strategy/tactical games (Stellaris, XCOM 2, Warhammer
40,000: Mechanicus, They Are Billions) -- a *weaker* genre match, not a franchise-biased one.
GTA V is similarly mixed: Saints Row III is a strong `query_plus_desc` hit, but Frostpunk /
Urban Empire / Kenshi are not close matches.

**Unexpected Results:** The predicted self-franchise/company bias never appeared, but
"`query_plus_desc` is uniformly more genre-coherent" -- the read from an earlier, smaller
exploratory pass -- does not hold on this sample either: 3/5 clear wins, 1/5 a clear regression,
1/5 mixed. Net effect still favors `query_plus_desc`, matching its aggregate Hit@K edge, but it
is not a clean sweep -- treat as example-dependent.

# Recommendation / Next Steps

**Recommended Action:** Keep `rag_chunk_v1_query_plus_desc` as the Ablation B default going
forward -- the specific franchise/marketing bias this notebook checked for was not found. Both
RAG arms still trail `two_tower_v1` on primary Slice A/B metrics (`eval_retrieval_overall.csv`,
`eval_retrieval_by_slice.csv`), so this remains a research finding, not a promotion decision.

**Risks:** Sample is n=5, hand-picked for maximum top-10 divergence among franchise/company-tagged
query games -- not a random or exhaustive sample. The NieR:Automata regression shows the
description signal can push retrieval toward a *different but still legitimate* genre neighborhood
that happens to be a worse match -- a subtler risk than the franchise-bias hypothesis this
notebook was built to test, and one this sample can't rule out at scale.

**Follow-up Analyses:** A larger random sample (~30-50 examples) if this arm is considered for
shipping; a franchise-ID-repeat-rate metric computed automatically across the full 12,500-example
cohort, rather than manual read only; a genre-overlap-with-query metric (automatable, unlike
franchise-repeat) to quantify the win/regression split seen here instead of relying on manual read.

**Open Questions:** Whether the same pattern holds at k=100 (full retrieval pool), not just the
top-5 shown here. Whether the `two_tower_v1` gap (still ~1.5-8% relative on primary metrics, see
Quantitative Recap) closes with embedder swap / blend-weight tuning / other pooling variants.